In [ ]:
import numpy as np
import string

# 1. Setup raw variables
TEXT = ("RAG pipelines retrieve context before generation. "
        "Chunking splits text into overlapping windows. "
        "Embeddings map text into vector space. "
        "Cosine similarity ranks chunks against the query. ")

QUERY = "chunking overlapping windows embeddings"

# Split text into words, convert to lowercase, and remove punctuation
words = TEXT.lower().split()
translator = str.maketrans('', '', string.punctuation)
words = [word.translate(translator) for word in words]

words

In [ ]:
# 2. Create chunks with sliding windows (Size = 8, Overlap = 3, Step = 5)
def chunk_text(word_list, size=8, overlap=3):
    chunks = []
    step = size - overlap
    for i in range(0, len(word_list), step):
        window = word_list[i : i + size]
        chunks.append(" ".join(window))
        if i + size >= len(word_list):
            break
    return chunks

chunks = chunk_text(words, size=8, overlap=3)
chunks

In [ ]:
# 3. Create vocabulary for Bag of Words (BoW)
vocabulary = sorted(list(set(words)))
vocabulary

In [ ]:
# 4. Generate embeddings (Vectorize texts based on word frequencies)
def vectorize(text_str, vocab):
    text_words = text_str.split()
    return [text_words.count(word) for word in vocab]

chunk_embeddings = [vectorize(c, vocabulary) for c in chunks]
query_embedding = vectorize(QUERY, vocabulary)

print("--- Chunk Embeddings ---")
for idx, emb in enumerate(chunk_embeddings):
    print(f"Chunk {idx}: {emb}")

print("\n--- Query Embedding ---")
print(query_embedding)


In [ ]:
# 5. Do something obvious: Rank chunks by Cosine Similarity against the Query
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm_1 = np.linalg.norm(v1)
    norm_2 = np.linalg.norm(v2)
    return dot / (norm_1 * norm_2) if norm_1 and norm_2 else 0.0

scores = [cosine_similarity(query_embedding, c_emb) for c_emb in chunk_embeddings]
best_idx = np.argmax(scores)

# Execution Summary
print("--- Extracted Text Chunks ---")
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx}: \"{chunk}\"")

print("\n--- Similarity Match Rankings ---")
for idx, score in enumerate(scores):
    print(f"Chunk {idx} Score: {score:.4f}")

print(f"\n🏆 Top Retrieved Context: \"{chunks[best_idx]}\"")
